## Problem Statement - To apply the artificial immune pattern recognition to perform a task of structure damage Classification.

**Sidhi Pawar BE B - 64**

In [17]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# ==========================================
# Steps 1, 2 & 3: Understanding, Preprocessing, and Representation
# ==========================================
def load_and_preprocess_data():
    print("Loading and Preprocessing Data (Step 2)...")
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
    df = pd.read_csv(url)
    y = df['Machine failure'].values # Target labels
    X = df.drop(columns=['UDI', 'Product ID', 'Type', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']).values
    
    # Normalization (Crucial for distance/affinity calculations)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, y

In [19]:

# ==========================================
# The Complete AIPR Class
# ==========================================
class EvolvedAIPR:
    def __init__(self, num_antibodies=200, clone_rate=5, mutation_rate=0.1, epochs=3):
        self.num_antibodies = num_antibodies
        self.clone_rate = clone_rate
        self.mutation_rate = mutation_rate
        self.epochs = epochs
        self.antibodies = []
        self.labels = []

    def train(self, X, y):
        # Step 4: Generation of Antibodies
        print("Initializing Antibodies (Step 4)...")
        initial_indices = np.random.choice(len(X), self.num_antibodies, replace=False)
        self.antibodies = X[initial_indices].copy()
        self.labels = y[initial_indices].copy()

        print("Starting Affinity Maturation and Training (Steps 5 & 6)...")
        # Step 6: Training Loop
        for epoch in range(self.epochs):
            for i, antigen in enumerate(X):
                antigen_label = y[i]
                
                # Only compare with antibodies designed for this specific class
                mask = (self.labels == antigen_label)
                if not np.any(mask): continue
                
                valid_antibodies = self.antibodies[mask]
                valid_indices = np.where(mask)[0]
                
                # Calculate Affinity (Euclidean Distance - lower is better)
                distances = np.linalg.norm(valid_antibodies - antigen, axis=1)
                best_match_idx = valid_indices[np.argmin(distances)]
                best_antibody = self.antibodies[best_match_idx]
                
                # Step 5: Affinity Maturation (Cloning and Mutation)
                # 1. Clone the best antibody
                clones = np.tile(best_antibody, (self.clone_rate, 1))
                
                # 2. Mutate the clones slightly
                noise = np.random.randn(*clones.shape) * self.mutation_rate
                mutated_clones = clones + noise
                
                # 3. Selection: Did any mutation improve the affinity?
                clone_distances = np.linalg.norm(mutated_clones - antigen, axis=1)
                best_clone_idx = np.argmin(clone_distances)
                
                # If a mutated clone is a better match than the original antibody, replace it
                if clone_distances[best_clone_idx] < np.min(distances):
                    self.antibodies[best_match_idx] = mutated_clones[best_clone_idx]
                    
    def predict(self, X):
        # Step 7: Classification
        print("Classifying test instances (Step 7)...")
        predictions = []
        for antigen in X:
            # Find the closest antibody in our evolved pool
            distances = np.linalg.norm(self.antibodies - antigen, axis=1)
            best_match_idx = np.argmin(distances)
            predictions.append(self.labels[best_match_idx])
        return np.array(predictions)

In [20]:

# ==========================================
# Execution and Step 8: Evaluation
# ==========================================
if __name__ == "__main__":
    X, y = load_and_preprocess_data()
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Initialize the evolved system
    aipr = EvolvedAIPR(num_antibodies=300, clone_rate=5, mutation_rate=0.05, epochs=2)
    
    # Train it (This will take slightly longer now due to the evolutionary loop)
    aipr.train(X_train, y_train)

    # Predict
    predictions = aipr.predict(X_test)
    
    # Step 8: Evaluation
    print("\n--- AIPR EVALUATION (Step 8) ---")
    accuracy = accuracy_score(y_test, predictions)
    print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")
    print(classification_report(y_test, predictions, target_names=["Intact (0)", "Damaged (1)"]))


Loading and Preprocessing Data (Step 2)...
Initializing Antibodies (Step 4)...
Starting Affinity Maturation and Training (Steps 5 & 6)...
Classifying test instances (Step 7)...

--- AIPR EVALUATION (Step 8) ---
Overall Accuracy: 96.30%

              precision    recall  f1-score   support

  Intact (0)       0.97      0.99      0.98      1932
 Damaged (1)       0.43      0.28      0.34        68

    accuracy                           0.96      2000
   macro avg       0.70      0.63      0.66      2000
weighted avg       0.96      0.96      0.96      2000

